# My Learning Journey: PyTorch Buffers

**Date**: February 21, 2026

## What confused me

When I looked at the `CausalAttention` class in ch03, I noticed `self.register_buffer("mask", ...)` instead of a plain `self.mask = ...`. I didn't understand why. This notebook is my attempt to figure that out by breaking things on purpose.

## My questions going in

- What exactly is a buffer? How is it different from a parameter?
- Why not just use a regular attribute?
- What goes wrong without `register_buffer`?

**Attribution**: Concepts from *Build a Large Language Model From Scratch* (Raschka), Ch03 bonus material. Rebuilt in my own words.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(123)
print(f"PyTorch: {torch.__version__}")

## Part 1: Without buffers — the hidden bug

Here's a `CausalAttention` that stores the mask as a plain attribute (`self.mask = ...`). It works fine on CPU, but the bug shows up on GPU.

In [ ]:
class CausalAttentionWithoutBuffers(nn.Module):
    """Causal attention with mask stored as a plain attribute (NOT a buffer)."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Plain attribute — NOT registered as a buffer
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, _ = x.shape
        q = self.W_query(x)
        k = self.W_key(x)
        v = self.W_value(x)
        scores = q @ k.transpose(1, 2)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        weights = torch.softmax(scores / k.shape[-1]**0.5, dim=-1)
        weights = self.dropout(weights)
        return weights @ v


# Test on CPU — works fine
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64], [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10], [0.05, 0.80, 0.55]]
)
batch = torch.stack((inputs, inputs), dim=0)  # (2, 6, 3)
d_in, d_out, ctx_len = 3, 2, batch.shape[1]

ca_no_buffer = CausalAttentionWithoutBuffers(d_in, d_out, ctx_len, dropout=0.0)

with torch.no_grad():
    out = ca_no_buffer(batch)
print("CPU output (no buffer):\n", out)

CPU works. Now let's see what happens when we move to GPU.

In [ ]:
# Detect available device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Running on: {device}")

# Move batch and model to device
batch_dev = batch.to(device)
ca_no_buffer.to(device)

# Check where each tensor actually lives
print("W_query.weight device:", ca_no_buffer.W_query.weight.device)
print("mask device:          ", ca_no_buffer.mask.device)

**I noticed**: `W_query` moved to the GPU, but `mask` stayed on CPU. That's the bug. If we run the forward pass on GPU with a CPU mask, PyTorch will error on the device mismatch.

The manual fix is tedious but works:

In [ ]:
# Manually move the mask — easy to forget, easy to get wrong
ca_no_buffer.mask = ca_no_buffer.mask.to(device)
print("mask device after manual move:", ca_no_buffer.mask.device)

with torch.no_grad():
    out_dev = ca_no_buffer(batch_dev)
print("GPU output (manual mask move):\n", out_dev.cpu())

## Part 2: With buffers — the clean solution

`register_buffer` tells PyTorch: "this tensor is part of the module but not a trainable parameter". PyTorch will:
- Move it automatically when `.to(device)` is called
- Include it in `state_dict` for serialization

In [ ]:
class CausalAttentionWithBuffer(nn.Module):
    """Causal attention with mask registered as a buffer — the correct approach."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register_buffer: not a parameter (no grad), but travels with the model
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, _ = x.shape
        q = self.W_query(x)
        k = self.W_key(x)
        v = self.W_value(x)
        scores = q @ k.transpose(1, 2)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        weights = torch.softmax(scores / k.shape[-1]**0.5, dim=-1)
        weights = self.dropout(weights)
        return weights @ v


ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, ctx_len, dropout=0.0)
ca_with_buffer.to(device)  # mask moves automatically!

print("W_query.weight device:", ca_with_buffer.W_query.weight.device)
print("mask device:          ", ca_with_buffer.mask.device)  # same as W_query

with torch.no_grad():
    out_buf = ca_with_buffer(batch_dev)
print("\nGPU output (with buffer):\n", out_buf.cpu())

## Part 3: Buffers and state_dict

A second advantage: buffers appear in `state_dict`, so they're saved and restored with the model checkpoint.

In [ ]:
print("Keys in state_dict WITHOUT buffer:")
for key in ca_no_buffer.state_dict().keys():
    print(" ", key)

print("\nKeys in state_dict WITH buffer:")
for key in ca_with_buffer.state_dict().keys():
    print(" ", key)  # 'mask' appears here

## My takeaways

| | Plain attribute | `register_buffer` |
|---|---|---|
| Auto-moves with `.to(device)` | ❌ No | ✅ Yes |
| Included in `state_dict` | ❌ No | ✅ Yes |
| Updated during training | ❌ No | ❌ No |
| Has gradients | ❌ No | ❌ No |

**Buffer vs Parameter**:
- `nn.Parameter`: trainable — updated by optimizer, has gradient
- Buffer: fixed — not updated, no gradient, but device-aware

**My rule of thumb**: If a tensor in a module is not trainable but needs to travel to GPU with the model, register it as a buffer. The causal mask is the perfect example: it's fixed (triangular matrix of 0s and 1s), never changes during training, but must be on the same device as the inputs.